In [1]:
import torch
from torch import nn
from torch.nn import functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        if use_1x1conv:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, X):
        Y = self.conv1(X)
        Y = self.bn1(Y)
        Y = F.relu(Y)

        Y = self.conv2(Y)
        Y = self.bn2(Y)

        X = self.shortcut(X)

        Y = Y + X
        
        return F.relu(Y)


In [2]:
def make_resnet_stage(in_channels, out_channels, num_blocks, first_stage=False):
    blocks = []

    for i in range(num_blocks):
        if i == 0 and not first_stage:
            blocks.append(
                ResidualBlock(
                    in_channels,
                    out_channels,
                    stride=2,
                    use_1x1conv=True
                )
            )
        else:
            blocks.append(
                ResidualBlock(
                    out_channels,
                    out_channels,
                    stride=1,
                    use_1x1conv=False
                )
            )
    
    return nn.Sequential(*blocks)



In [3]:
stem = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU()
)

stage1 = make_resnet_stage(64, 64, 2, first_stage=True)
stage2 = make_resnet_stage(64, 128, 2)
stage3 = make_resnet_stage(128, 256, 2)
stage4 = make_resnet_stage(256, 512, 2)

net = nn.Sequential(
    stem,
    stage1,
    stage2,
    stage3,
    stage4,
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(512, 10)
)

In [4]:
from pathlib import Path
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

DATA_DIR = Path("CIFAR-10")

if not (DATA_DIR / "trainLabels.csv").exists():
    DATA_DIR = Path(".")

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
LABEL_CSV = DATA_DIR / "trainLabels.csv"

In [5]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = val_transform

In [6]:
classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

class_to_idx = {name: i for i, name in enumerate(classes)}
idx_to_class = {i: name for name, i in class_to_idx.items()}

In [7]:
class PngDataset(Dataset):
    def __init__(self, img_paths, transform=None, labels=None):
        self.img_paths = list(img_paths)
        self.transform = transform
        self.labels = labels
        if self.labels is not None and len(self.img_paths) != len(self.labels):
            raise ValueError("img_paths and labels must have the same length")
    
    def __len__(self):
        return len(self.img_paths)
    
    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)
        
        if self.labels is None:
            return image, img_path.name
        
        label = int(self.labels[idx])
        return image, label

In [8]:
label_df = pd.read_csv(LABEL_CSV)
label_df["label_idx"] = label_df["label"].map(class_to_idx)

if label_df["label_idx"].isna().any():
    unknown = sorted(label_df.loc[label_df["label_idx"].isna(), "label"].unique())
    raise ValueError(f"Unknown labels: {unknown}")

all_img_paths = [TRAIN_DIR / f"{img_id}.png" for img_id in label_df["id"]]
all_labels = label_df["label_idx"].astype(int).tolist()

missing = [path for path in all_img_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing training images, for example: {missing[:5]}")

test_img_paths = sorted(TEST_DIR.glob("*.png"), key=lambda path: int(path.stem))

print(f"train images: {len(all_img_paths)}")
print(f"test images: {len(test_img_paths)}")

train images: 50000
test images: 300000


In [9]:
SEED = 42
VAL_RATIO = 0.1

generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(all_img_paths), generator=generator).tolist()
val_size = int(len(indices) * VAL_RATIO)

val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_img_paths = [all_img_paths[i] for i in train_indices]
train_labels = [all_labels[i] for i in train_indices]
val_img_paths = [all_img_paths[i] for i in val_indices]
val_labels = [all_labels[i] for i in val_indices]

train_dataset = PngDataset(train_img_paths, labels=train_labels, transform=train_transform)
val_dataset = PngDataset(val_img_paths, labels=val_labels, transform=val_transform)
test_dataset = PngDataset(test_img_paths, labels=None, transform=test_transform)

print(f"train split: {len(train_dataset)}")
print(f"val split: {len(val_dataset)}")
print(f"test split: {len(test_dataset)}")

train split: 45000
val split: 5000
test split: 300000


In [10]:
BATCH_SIZE = 128
NUM_WORKERS = 0

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = net.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    net.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4
)
EPOCHS = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"using device: {device}")

using device: cuda


In [12]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size

    return running_loss / total, correct / total


best_acc = 0.0
best_model_path = DATA_DIR / "best_resnet_cifar10.pth"

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(net, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(net, val_loader, criterion, device)
    scheduler.step()

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(net.state_dict(), best_model_path)

    print(
        f"epoch {epoch:02d}/{EPOCHS} "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
        f"best_val_acc={best_acc:.4f}"
    )

epoch 01/20 train_loss=2.0444 train_acc=0.2649 val_loss=1.6508 val_acc=0.3854 best_val_acc=0.3854
epoch 02/20 train_loss=1.5620 train_acc=0.4213 val_loss=1.4764 val_acc=0.4578 best_val_acc=0.4578
epoch 03/20 train_loss=1.3268 train_acc=0.5166 val_loss=1.2364 val_acc=0.5662 best_val_acc=0.5662
epoch 04/20 train_loss=1.1209 train_acc=0.5980 val_loss=1.0373 val_acc=0.6286 best_val_acc=0.6286
epoch 05/20 train_loss=0.9614 train_acc=0.6580 val_loss=0.8591 val_acc=0.7042 best_val_acc=0.7042
epoch 06/20 train_loss=0.8437 train_acc=0.7036 val_loss=0.8097 val_acc=0.7152 best_val_acc=0.7152
epoch 07/20 train_loss=0.7348 train_acc=0.7412 val_loss=0.7591 val_acc=0.7348 best_val_acc=0.7348
epoch 08/20 train_loss=0.6373 train_acc=0.7773 val_loss=0.6658 val_acc=0.7656 best_val_acc=0.7656
epoch 09/20 train_loss=0.5674 train_acc=0.8027 val_loss=0.7695 val_acc=0.7556 best_val_acc=0.7656
epoch 10/20 train_loss=0.5133 train_acc=0.8243 val_loss=0.6258 val_acc=0.7930 best_val_acc=0.7930
epoch 11/20 train_lo

In [13]:
@torch.no_grad()
def predict_test(model, loader, device):
    model.eval()
    rows = []

    for images, names in loader:
        images = images.to(device, non_blocking=True)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().tolist()

        for name, pred in zip(names, preds):
            rows.append({"id": int(Path(name).stem), "label": idx_to_class[pred]})

    return pd.DataFrame(rows).sort_values("id")


net.load_state_dict(torch.load(best_model_path, map_location=device))
submission = predict_test(net, test_loader, device)
submission_path = DATA_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print(submission.head())
print(f"saved to {submission_path}")

   id       label
0   1        deer
1   2    airplane
2   3  automobile
3   4        ship
4   5        bird
saved to submission.csv
